# Basketball: Sports Analytics

**Sports Analytics Club at UT Dallas · Workshop 2**

**Edit this cell** marks your tasks. **Run this cell** means the code is ready; leave it as provided.
Work from top to bottom and replace `____` using the example.

> **Sport in 30 seconds**
> - One row = one NBA player’s 2023–24 regular season, combining teams.
> - Box-score stats record actions such as scoring, passing, and defending.
> - We group similar playing profiles; there is no outcome to predict.

## Setup

Connect to a Colab **CPU** runtime and run both collapsed cells. The dataset is included and its checksum is checked.

In [ ]:
#@title Import libraries — run this cell
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

SEED = 42
plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


In [ ]:
#@title Prepared dataset — run this cell (expand only if curious)
# Prepared data snapshot. No network calls. Instructor rebuilds with scripts/prepare_data.py.
import base64, hashlib, io, zipfile
EXPECTED_SHA256 = {'nba_data.csv': 'a61fe35ad6b7f43489e9220fa88973c0f0c5642c632065e4d4ee33869a2efe50'}
SNAPSHOT_B64 = "..."
candidates = [Path('data'), Path('../data')]
DATA_DIR = next((p for p in candidates if all((p/n).is_file() for n in EXPECTED_SHA256)), None)
if DATA_DIR is None:
    DATA_DIR = Path('workshop2_data')
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(SNAPSHOT_B64))) as archive:
        for name in EXPECTED_SHA256:
            (DATA_DIR/name).write_bytes(archive.read(name))
for name, expected in EXPECTED_SHA256.items():
    actual = hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()
    assert actual == expected, f'{name} differs from the workshop snapshot. Restore the supplied CSV.'
print('Ready: 1 verified CSV.')


## First look

Run this cell to preview the data.

In [ ]:
players = pd.read_csv(DATA_DIR/'nba_data.csv')
for name, frame in [('NBA', players)]:
    print(name, 'rows/columns:', frame.shape, '| missing cells:', frame.isna().sum().sum())
    display(frame.head(3))


| Column | Plain English |
|---|---|
| `points_per36` | Points: points scored |
| `rebounds_per36` | Rebounds: gaining possession after a missed shot |
| `assists_per36` | Assists: passes leading directly to a basket |
| `steals_per36` | Steals: taking possession from the opponent |
| `blocks_per36` | Blocks: stopping or deflecting a shot attempt |
| `three_attempts_per36` | 3PA: three-point shots attempted |

**Per 36 minutes** puts players on a more comparable playing-time scale instead of using raw season totals. The sample includes players with at least 20 games and 500 minutes.

## K-Means: find similar profiles

`K` is the number of groups. Start with four, then try three.

In [ ]:
#@title Run this cell — scale the six stats
# Per-36 rates adjust for minutes, not pace or role. Scaling keeps large-range stats from dominating.
nba_features = ['points_per36', 'rebounds_per36', 'assists_per36',
                'steals_per36', 'blocks_per36', 'three_attempts_per36']
nba_scaler = StandardScaler()
nba_scaled = nba_scaler.fit_transform(players[nba_features])


### Worked example — run this cell

In [ ]:
# Worked example: make four groups and view the first five player labels.
example_clusterer = KMeans(n_clusters=4, random_state=SEED, n_init=10)
example_labels = example_clusterer.fit_predict(nba_scaled)
print(example_labels[:5])

### B1 · Edit this cell

**Predict first:** If we use fewer clusters, which types of players do you expect to get grouped together?

Start with `K = 4` and fit on `nba_scaled`. Then try `K = 3`. Each time, run your edit and the next output cell.

In [ ]:
K = ____
cluster_model = KMeans(n_clusters=int(K), random_state=SEED, n_init=10)
cluster_labels = cluster_model.fit_predict(____)


In [ ]:
#@title Run this cell — show groups, profiles, and player examples
clustered = players.assign(cluster=cluster_labels)
cluster_profiles = clustered.groupby('cluster')[nba_features].mean()
display(clustered.groupby('cluster').size().rename('players').to_frame())
display(cluster_profiles.round(2))

profile_z = pd.DataFrame(nba_scaled, columns=nba_features).assign(cluster=cluster_labels).groupby('cluster').mean()
fig, ax = plt.subplots(figsize=(10, 4.5))
limit = max(1, np.abs(profile_z.to_numpy()).max())
heat = ax.imshow(profile_z, cmap='RdBu_r', vmin=-limit, vmax=limit, aspect='auto')
ax.set_xticks(range(len(nba_features)), ['Points','Rebounds','Assists','Steals','Blocks','3PA'])
ax.set_yticks(range(int(K)), [f'Cluster {i}' for i in profile_z.index])
for i in range(len(profile_z)):
    for j in range(len(nba_features)):
        ax.text(j, i, f'{profile_z.iloc[i,j]:.1f}', ha='center', va='center')
fig.colorbar(heat, ax=ax, label='Average standardized value')
ax.set_title('NBA cluster profiles: above or below this sample average')
plt.tight_layout(); plt.show()

# Players closest to each cluster center, not the best players.
distances = cluster_model.transform(nba_scaled)
representatives = []
for cluster in range(int(K)):
    members = np.flatnonzero(cluster_labels == cluster)
    closest = members[np.argsort(distances[members, cluster])[:3]]
    representatives.append(clustered.iloc[closest][['cluster','player_name','teams','position']])
display(pd.concat(representatives, ignore_index=True))


**Describe one group using two of its stats.**

High rebounds + blocks often describes more interior/defensive players (near the basket); high 3PA often describes more perimeter-oriented players (farther from the basket).

Use the numbers; no player-name knowledge is needed. Group numbers are labels, not rankings or fixed positions.

## Optional: PCA

PCA compresses the same six stats into fewer axes. The map shows two axes; colors come from K-Means.

### B2 · Edit this cell

Try keeping two components, then three. Run your edit and the next output cell to compare retained variance.

In [ ]:
n_components_to_keep = 2   # Change to 3


In [ ]:
#@title Run this cell — show the PCA map and retained variance
# Worked example: fit all six axes so we can compare how much 2 or 3 retain.
pca = PCA(n_components=len(nba_features), svd_solver='full')
all_components = pca.fit_transform(nba_scaled)
pca_map = players[['player_name', 'teams']].copy()
pca_map['PC1'], pca_map['PC2'] = all_components[:,0], all_components[:,1]
pca_map['cluster'] = cluster_labels
explained = pca.explained_variance_ratio_
print(f'Two components retain {explained[:2].sum():.1%} of variance in these standardized features.')
print(f'The map leaves out {1-explained[:2].sum():.1%}.')

fig, ax = plt.subplots(figsize=(9, 5))
for cluster in sorted(pca_map.cluster.unique()):
    subset = pca_map.loc[pca_map.cluster == cluster]
    ax.scatter(subset.PC1, subset.PC2, label=f'Cluster {cluster}', alpha=0.65, s=24)
# Label one player to help read the map.
player_to_label = 'Stephen Curry'
chosen = pca_map.loc[pca_map.player_name.eq(player_to_label)]
if chosen.empty:
    print('Name not found. Search players["player_name"] and use the exact spelling.')
else:
    for row in chosen.itertuples():
        ax.annotate(row.player_name, (row.PC1,row.PC2), xytext=(5,8), textcoords='offset points')
ax.set(xlabel=f'PC1 ({explained[0]:.1%} of variance)', ylabel=f'PC2 ({explained[1]:.1%} of variance)',
       title='NBA profiles compressed to two dimensions')
ax.legend(); plt.tight_layout(); plt.show()

assert 1 <= n_components_to_keep <= len(nba_features)
print(f'{n_components_to_keep} components retain {explained[:n_components_to_keep].sum():.1%} of variance.')
plt.plot(range(1, len(nba_features)+1), np.cumsum(explained), marker='o', color='#157F78')
plt.axvline(n_components_to_keep, linestyle='--', color='gray')
plt.xlabel('Number of components kept'); plt.ylabel('Cumulative variance retained')
plt.ylim(0, 1.05); plt.xticks(range(1,len(nba_features)+1))
plt.title('More components retain more of the original variation')
plt.tight_layout(); plt.show()
display(pd.DataFrame(pca.components_[:2].T, index=nba_features,
                     columns=['PC1 weight','PC2 weight']).round(3))

print(f'Variance retained: 2 components = {explained[:2].sum():.1%}; 3 components = {explained[:3].sum():.1%}.')


**How much more variance do three components retain?**

The map stays 2D. Nearby players may differ in omitted components; retained variance is not a ranking or player-quality score.

<details>
<summary>Sources and definitions</summary>

Prepared files and complete column definitions: `data/DATA_SOURCES.md` and `data/DATA_DICTIONARY.md` in the package.
Raw-source URLs, checksums, and row-filter audit: `data/manifest.json`.

- **NBA:** [SportsDataverse ESPN player box scores](https://github.com/sportsdataverse/sportsdataverse-data/releases/tag/espn_nba_player_boxscores), `player_box_2024.csv`. NBA 2023–24 regular season only, excluding the All-Star game and In-Season Tournament final.

- **Methods:** [K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means) · [PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca) · [scaling](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

</details>